# SMRD Phase 2B.4 — Colab/T4 runner

This notebook executes the **real** pipeline on a GPU with at least 7 GiB of
usable VRAM. The local 6 GiB machine is only used for preprocessing, caching,
master creation, correctness probes and smoke tests.

Order (design §11.5):
1. environment probe (P1)
2. feature cache (P2 reference check)
3. master initial checkpoint
4. preflight P3/P4
5. gates G1–G6
6. Arm B / Twitter16 search + retrain
7. remaining arms and the headline report

Set `PROJECT_ROOT` to the folder that contains `src/` and `data/`.

In [ ]:
#@title 1. Install dependencies
import torch, sys
print('python', sys.version.split()[0])
print('torch', torch.__version__, 'cuda', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('Runtime -> Change runtime type -> T4 GPU')
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory / 1024**3, 'GiB')

%pip install -q 'transformers==5.22.1' 'tokenizers==0.23.3' 'safetensors==0.8.0' \
    'huggingface_hub==2.4.0' 'numpy==2.5.4' 'pandas==3.1.0' 'scikit-learn==1.9.1' \
    'nltk==3.10.4' 'tqdm==4.70.1' 'bitsandbytes==0.50.2'

In [ ]:
#@title 2. Locate project and download NLTK data
import os, pathlib

# Option A: project uploaded to Drive at MyDrive/smrd-reproduction-main
# Option B: set this to wherever you uploaded/cloned the repository
PROJECT_ROOT = pathlib.Path('/content/drive/MyDrive/smrd-reproduction-main')
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = pathlib.Path('/content/smrd-reproduction-main')
assert (PROJECT_ROOT / 'src' / 'run.py').exists(), (
    f'Project not found at {PROJECT_ROOT}. Upload the repository (including data/) first.'
)
os.chdir(PROJECT_ROOT)
print('project:', PROJECT_ROOT)

import nltk
for pkg in ('stopwords', 'wordnet', 'omw-1.4', 'punkt'):
    nltk.download(pkg, quiet=True)
print('NLTK data ready')

In [ ]:
#@title 3. P1 — hardware probe (requires >= 7.0 GiB usable VRAM)
!python -m src.run probe

In [ ]:
#@title 4. Feature caches + master checkpoint
!python -m src.run cache --all-datasets
!python -m src.run master

In [ ]:
#@title 5. Preflight + gates (P1–P4, G1–G6)
!python -m src.run preflight
!python -m src.run gates --include-stress

In [ ]:
#@title 6. Arm B / Twitter16 pilot (36 evaluations + final retrain)
!python -m src.run pilot

In [ ]:
#@title 7. Remaining runs and headline report
# Remove the leading '#' to run all nine arm/dataset combinations.
# !python -m src.run all
!python -m src.run report
print(open('reports/headline_comparison.md', encoding='utf-8').read())